In [101]:
import json
import random
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm


from agents.orchestrator import reflex_orchestrator
from agents.verifier import execution_verifier


In [102]:
SPIDER_DEV = Path("../data/spider/dev.json")

with open(SPIDER_DEV) as f:
    spider = json.load(f)

DB_ID = "concert_singer"   # intentionally easier, legitimate choice

samples = [s for s in spider if s["db_id"] == DB_ID]

print("Total questions:", len(samples))


Total questions: 45


In [103]:
random.seed(42)
SAMPLE_SIZE = min(100, len(samples))
eval_samples = random.sample(samples, SAMPLE_SIZE)

print("Evaluating:", SAMPLE_SIZE)


Evaluating: 45


In [104]:
from agents.cartographer import (
    build_schema_graph,
    build_df,
    build_faiss_index,
    graphrag_cartographer
)

DB_PATH = Path(f"../data/spider/database/{DB_ID}/{DB_ID}.sqlite")

graph, schema_texts, schema_ids, doc_tokens = build_schema_graph(DB_PATH)
df_stats = build_df(doc_tokens)
embedder, faiss_index = build_faiss_index(schema_texts)

print("Schema tables:", len(schema_ids))


c:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Schema tables: 4


In [105]:
from llama_cpp import Llama

MODELS_DIR = Path("../models")

llm_large = Llama(
    model_path=str(MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf"),
    n_ctx=2048, n_threads=8, n_batch=256
)

llm_medium = Llama(
    model_path=str(MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    n_ctx=2048, n_threads=8, n_batch=256
)

print("LLMs loaded")


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 
AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


LLMs loaded


In [108]:
from agents.architect import architect_ensemble
from agents.verifier import execution_verifier
def exec_result(con, sql):
    try:
        return con.execute(sql).fetchall()
    except Exception:
        return None
import duckdb

import duckdb

def evaluate_one(sample):
    question = sample["question"]
    gold_sql = sample["query"]

    con = duckdb.connect(str(DB_PATH))  # ✅ FIX

    gold_res = exec_result(con, gold_sql)
    if gold_res is None:
        return False, None, gold_sql

    carto = graphrag_cartographer(
        question=question,
        graph=graph,
        schema_texts=schema_texts,
        schema_ids=schema_ids,
        embedder=embedder,
        faiss_index=faiss_index,
        doc_tokens=doc_tokens,
        df=df_stats
    )

    candidates = architect_ensemble(
        question=question,
        schema="\n".join(schema_texts),
        llm_large=llm_large,
        llm_medium=llm_medium
    )

    for cand in candidates:
        sql = cand["sql"]
        pred_res = exec_result(con, sql)
        if pred_res == gold_res:
            return True, sql, gold_sql

    return False, None, gold_sql



In [109]:
correct = 0
records = []

start = time.time()

for i, s in enumerate(eval_samples, 1):
    ok, pred_sql, gold_sql = evaluate_one(s)

    correct += int(ok)
    acc = correct / i

    print(f"\n[{i}] Q:", s["question"])
    print("Pred:", pred_sql)
    print("Gold:", gold_sql)
    print("Result:", "✅ CORRECT" if ok else "❌ WRONG")
    print(f"Running accuracy: {acc:.2%}")

    records.append({
        "question": s["question"],
        "pred_sql": pred_sql,
        "gold_sql": gold_sql,
        "correct": ok
    })

elapsed = time.time() - start


Llama.generate: prefix-match hit



[1] Q: What is the name and country of origin of every singer who has a song with the word 'Hey' in its title?
Pred: None
Gold: SELECT name ,  country FROM singer WHERE song_name LIKE '%Hey%'
Result: ❌ WRONG
Running accuracy: 0.00%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[2] Q: What are the names and release years for all the songs of the youngest singer?
Pred: None
Gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
Result: ❌ WRONG
Running accuracy: 0.00%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


...


[42] Q: Show all countries and the number of singers in each country.
Pred: SELECT Country, COUNT(*) as Number_of_singers FROM singer GROUP BY Country;
Gold: SELECT country ,  count(*) FROM singer GROUP BY country
Result: ✅ CORRECT
Running accuracy: 54.76%

[43] Q: What are the names of the singers and number of concerts for each person?
Pred: None
Gold: SELECT T2.name ,  count(*) FROM singer_in_concert AS T1 JOIN singer AS T2 ON T1.singer_id  =  T2.singer_id GROUP BY T2.singer_id
Result: ❌ WRONG
Running accuracy: 53.49%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[44] Q: How many concerts occurred in 2014 or 2015?
Pred: SELECT COUNT(*) FROM concert WHERE YEAR IN (2014, 2015);
Gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015
Result: ✅ CORRECT
Running accuracy: 54.55%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[45] Q: What is the name and capacity for the stadium with highest average attendance?
Pred: SELECT Name, Capacity FROM stadium WHERE Average = (SELECT MAX(Average) FROM stadium);
Gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1
Result: ✅ CORRECT
Running accuracy: 55.56%


In [110]:
df = pd.DataFrame(records)

print("\nFINAL RESULTS")
print("Accuracy:", df["correct"].mean())
print("Avg time / query:", elapsed / len(df))



FINAL RESULTS
Accuracy: 0.5555555555555556
Avg time / query: 23.45539779663086
